# ShiftGuard-X — NB93: Certified Conditional-Dependency Feasibility Gate

**Research status:** An experimental feasibility test, **not** an established new compressor or originality claim.

This notebook tests whether an exact IEEE-754 representation can exploit *partial* computational relationships while recording exceptions **and every byte of metadata**. Competitors include frozen-calibration univariate Zstandard/LZMA/PCodec, whole-matrix Zstandard, XOR residual coding, and a straightforward formula+exception bitmap (closest simple prior-art mechanism).

**Two independent real public datasets:** UCI Bike Sharing (exact relationship `cnt = casual + registered`) and NYC TLC January 2024 Yellow Taxi (monetary total potentially differs from the sum of reported component charges). These are **development/positive-control datasets**, not unseen novelty confirmation. For the taxi dataset, the `total_amount` formula is a *hypothesis*, not assumed to be correct. If datasets fail to download, the resulting archive is **INCOMPLETE**; never replace missing real data with synthetic measurements.

**Pre-registered gates:** (1) exact bitwise decode, signed zero/NaN payloads/truncation controls; (2) no evaluation-time codec/mode selection; (3) >=5% held-out size improvement versus strongest tested baseline (including a conservative **post-test comparator oracle** for two fixed joint layouts) in each of two independent families, no >10% regression; (4) <= a few seconds selection cost per tested stream where feasible; (5) an explicit distinct coding rule that does **not** reduce to ordinary functional-dependency elimination, XOR, or exception masks. **Passing 1–4 does not establish originality: condition 5 needs separate literature/source examination.**

Important prior-art exclusions: [Corra](https://arxiv.org/abs/2403.17229), [Virtual](https://arxiv.org/abs/2410.14066), [Derived-Column Elimination (Zenodo, 2026)](https://doi.org/10.5281/zenodo.20848308), [PCodec](https://arxiv.org/abs/2502.06112), [ALP](https://ir.cwi.nl/pub/33334/). The 2026 exact-dependency preprint in particular prevents claiming ordinary formula substitution as new.

Run in a **fresh Colab CPU runtime** (`Runtime → Run all`). Download the resulting `SGX_NB93_results.zip`, even if marked INCOMPLETE, and share it for review.


In [ ]:
# Install version-pinned dependencies in Colab; explicitly record any failures.
import sys, subprocess, importlib.util
required = {'zstandard':'0.25.0','pcodec':'1.0.3','pyarrow':'20.0.0','pandas':'2.2.2','numpy':'2.0.2'}
INSTALL = {}
for package, version in required.items():
    if importlib.util.find_spec(package) is not None:
        INSTALL[package] = 'already available'
    else:
        p = subprocess.run([sys.executable,'-m','pip','-q','install',f'{package}=={version}'],text=True,capture_output=True)
        INSTALL[package] = {'returncode':p.returncode,'tail':p.stderr[-350:]}
print(INSTALL)


In [ ]:
import os, io, csv, json, time, lzma, struct, hashlib, zipfile, urllib.request, platform, traceback
from pathlib import Path
import numpy as np, pandas as pd
import zstandard as zstd
try:
    import pcodec
    from pcodec import standalone as pcs
except Exception:
    pcodec=pcs=None
import pyarrow.parquet as pq
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUT=ROOT/'SGX_NB93_outputs'; OUT.mkdir(exist_ok=True)
CACHE=ROOT/'SGX_NB93_source_cache'; CACHE.mkdir(exist_ok=True)
ERR=[]; DATA=[]; PROVENANCE=[]
UCI_URL='https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
NYC_URL='https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
MAX_TAXI_ROWS=120000

def download_exact(url, dest):
    if not dest.exists() or dest.stat().st_size == 0:
        req=urllib.request.Request(url,headers={'User-Agent':'ShiftGuardX-Research/93 (noncommercial academic benchmark)'})
        with urllib.request.urlopen(req,timeout=100) as src, open(dest,'wb') as dst:
            while True:
                b=src.read(1024*1024)
                if not b:break
                dst.write(b)
    b=dest.read_bytes()
    return hashlib.sha256(b).hexdigest(),len(b)

def load_bike():
    path=CACHE/'bike_sharing_uci275.zip'
    sha,sz=download_exact(UCI_URL,path)
    with zipfile.ZipFile(path) as z:
        member=next(s for s in z.namelist() if s.lower().endswith('hour.csv'))
        df=pd.read_csv(io.BytesIO(z.read(member)))
    names=['casual','registered','cnt']
    assert all(s in df.columns for s in names)
    m={s:np.ascontiguousarray(df[s].to_numpy(dtype='<f8')) for s in names}
    PROVENANCE.append({'family':'UCI-Bike-275','url':UCI_URL,'sha256':sha,'source_bytes':sz,'n_rows':len(df),'source_member':member,'transform':'Original integer-valued columns converted exactly to IEEE float64; not CSV-byte restoration'})
    return {'family':'UCI-Bike-275','sources':names[:-1], 'target':names[-1], 'values':m}

def load_taxi():
    path=CACHE/'yellow_tripdata_2024-01.parquet'
    sha,sz=download_exact(NYC_URL,path)
    pqfile=pq.ParquetFile(path)
    available={s.lower():s for s in pqfile.schema_arrow.names}
    required=['fare_amount','extra','mta_tax','tip_amount','tolls_amount','improvement_surcharge','congestion_surcharge','airport_fee','total_amount']
    missing=[c for c in required if c not in available]
    if missing:raise ValueError('Source Parquet missing required fields: '+str(missing))
    actual=[available[k] for k in required]
    # Bounded memory: only columns needed, and stop after first MAX_TAXI_ROWS.
    tables=[];nrows=0
    for batch in pqfile.iter_batches(batch_size=25000,columns=actual):
        if nrows>=MAX_TAXI_ROWS:break
        b=batch.to_pandas().iloc[:MAX_TAXI_ROWS-nrows]
        tables.append(b);nrows+=len(b)
    df=pd.concat(tables,ignore_index=True)
    m={canon:np.ascontiguousarray(pd.to_numeric(df[available[canon]],errors='coerce').to_numpy(dtype='<f8')) for canon in required}
    PROVENANCE.append({'family':'NYC-TLC-Yellow-2024-01','url':NYC_URL,'sha256':sha,'source_bytes':sz,'n_rows':len(df),'source_member':'first 120000 parquet rows; selected numeric columns','transform':'Parquet-decoded float64 bit patterns; null conversion to numpy NaN, not original parquet-file bytes'})
    return {'family':'NYC-TLC-Yellow-2024-01','sources':required[:-1], 'target':required[-1], 'values':m}

for label,fn in [('UCI-Bike-275',load_bike),('NYC-TLC-Yellow-2024-01',load_taxi)]:
    try:
        d=fn();DATA.append(d)
        print(label,'loaded rows:',len(d['values'][d['target']]))
    except Exception as e:
        ERR.append({'family':label,'stage':'data_loading','error':repr(e)})
        print(label,'INCOMPLETE:',repr(e))


In [ ]:
# Complete-file coder; every method carries its decoding metadata.
MAGIC=b'SG93'; C=zstd.ZstdCompressor(level=3);D=zstd.ZstdDecompressor()

def as_f8(arr):return np.asarray(arr,dtype='<f8').copy()
def fbits(arr):return as_f8(arr).view('<u8')
def as_u8(data):return np.frombuffer(data,dtype='<u8').copy()

def split8(b):
    x=np.frombuffer(b,dtype=np.uint8)
    if len(x)%8:raise ValueError('nonaligned byte stream')
    return x.reshape(-1,8).T.copy().tobytes()
def unsplit8(b):
    x=np.frombuffer(b,dtype=np.uint8)
    if len(x)%8:raise ValueError('nonaligned split stream')
    return x.reshape(8,-1).T.copy().tobytes()

def enc_bytes(raw,codec):
    if codec=='zstd':return C.compress(raw)
    if codec=='zstd_split':return C.compress(split8(raw))
    if codec=='lzma':return lzma.compress(raw,preset=1)
    raise ValueError(codec)
def dec_bytes(payload,codec):
    if codec=='zstd':return D.decompress(payload)
    if codec=='zstd_split':return unsplit8(D.decompress(payload))
    if codec=='lzma':return lzma.decompress(payload)
    raise ValueError(codec)

def enc_column(v,method):
    if method=='pcodec':
        if pcs is None:raise RuntimeError('pcodec unavailable')
        return bytes(pcs.simple_compress(as_f8(v),pcodec.ChunkConfig()))
    return enc_bytes(as_f8(v).tobytes(),method)
def dec_column(p,method,n):
    if method=='pcodec':
        if pcs is None:raise RuntimeError('pcodec unavailable')
        v=as_f8(pcs.simple_decompress(p))
    else:
        raw=dec_bytes(p,method)
        if len(raw)!=8*n:raise ValueError('wrong decoded byte count')
        v=np.frombuffer(raw,dtype='<f8').copy()
    if len(v)!=n:raise ValueError('wrong decoded element count')
    return v

METHODS=['zstd','zstd_split','lzma']+(['pcodec'] if pcs is not None else [])
def frozen_codec(calib):
    outcomes=[]
    for method in METHODS:
        try:
            t=time.perf_counter();p=enc_column(calib,method)
            assert dec_column(p,method,len(calib)).tobytes()==as_f8(calib).tobytes()
            outcomes.append((len(p),method,time.perf_counter()-t))
        except Exception as e:ERR.append({'stage':'codec_trial','codec':method,'error':repr(e)})
    if not outcomes:raise RuntimeError('no bit-exact compressor available')
    return min(outcomes)[1],outcomes

def predictors(values,sources):
    n=len(values[sources[0]])
    with np.errstate(all='ignore'):
        raw=np.zeros(n,dtype='<f8')
        for name in sources:raw=np.add(raw,values[name],dtype='<f8')
        cents=np.zeros(n,dtype=np.int64)
        valid=np.ones(n,dtype=bool)
        for name in sources:
            v=as_f8(values[name])
            okay=np.isfinite(v)&(np.abs(v)<1e12)
            valid &= okay
            vv=np.where(okay,v,0)
            cents+=np.rint(vv*100.0).astype(np.int64)
        cents_pred=cents.astype('<f8')/100.0
        cents_pred[~valid]=np.nan
    return raw,cents_pred

def masks(values,sources,target,mode):
    truth=fbits(values[target]); a,b=predictors(values,sources)
    a_bits=fbits(a);b_bits=fbits(b)
    if mode=='mask_raw':
        code=np.where(truth==a_bits,0,2).astype(np.uint8)
    elif mode=='mask_cents':
        code=np.where(truth==b_bits,0,2).astype(np.uint8)
    elif mode=='mask_tri':
        code=np.where(truth==a_bits,0,np.where(truth==b_bits,1,2)).astype(np.uint8)
    else:raise ValueError(mode)
    return code

def pack_mode(code,tri):
    if tri:
        n=len(code);pad=(-n)%4
        if pad:code=np.pad(code,(0,pad),constant_values=3)
        x=code.reshape(-1,4)
        return ((x[:,0]<<6)|(x[:,1]<<4)|(x[:,2]<<2)|x[:,3]).astype('u1').tobytes()
    return np.packbits(code==2,bitorder='big').tobytes()

def unpack_mode(raw,n,tri):
    if tri:
        x=np.frombuffer(raw,dtype='u1');out=np.empty(len(x)*4,dtype='u1')
        out[0::4]=x>>6;out[1::4]=(x>>4)&3;out[2::4]=(x>>2)&3;out[3::4]=x&3
        return out[:n].copy()
    return (np.unpackbits(np.frombuffer(raw,dtype='u1'),bitorder='big')[:n]*2).astype('u1')

def table_hash(values,names):
    h=hashlib.sha256()
    for name in names:h.update(as_f8(values[name]).tobytes())
    return h.hexdigest()

def frame_pack(meta,chunks):
    assert len(meta['chunks'])==len(chunks)
    m=json.dumps(meta,sort_keys=True,separators=(',',':'),allow_nan=False).encode('utf8')
    return MAGIC+struct.pack('<I',len(m))+m+b''.join(chunks)

def frame_unpack(p):
    if len(p)<8 or p[:4]!=MAGIC:raise ValueError('frame truncated or invalid')
    mlen=struct.unpack_from('<I',p,4)[0]
    if mlen>1024*1024 or len(p)<8+mlen:raise ValueError('truncated metadata')
    meta=json.loads(p[8:8+mlen]);pos=8+mlen;chunks=[]
    for info in meta['chunks']:
        length=int(info['nbytes'])
        if length<0 or pos+length>len(p):raise ValueError('truncated chunk')
        chunks.append(p[pos:pos+length]);pos+=length
    if pos!=len(p):raise ValueError('trailing frame bytes')
    return meta,chunks

def encode(values,sources,target,mode,frozen_methods):
    n=len(values[target]); names=sources+[target];parts=[];tags=[]
    def add(kind,p):
        tags.append({'kind':kind,'nbytes':len(p)});parts.append(p)
    for src in sources:add('col:'+src,enc_column(values[src],frozen_methods[src]))
    targetv=as_f8(values[target]);truth=targetv.view('<u8')
    if mode=='raw':
        add('target',enc_column(targetv,frozen_methods[target]))
    elif mode in ('mask_raw','mask_cents','mask_tri'):
        code=masks(values,sources,target,mode)
        add('mask',C.compress(pack_mode(code,mode=='mask_tri')))
        add('exceptions',C.compress(truth[code==2].astype('<u8').tobytes()))
    elif mode=='xor_raw':
        pred,_=predictors(values,sources)
        add('xor',C.compress(split8((truth^fbits(pred)).astype('<u8').tobytes())))
    else:raise ValueError(mode)
    meta={'schema':'SG93-v1','n':n,'sources':sources,'target':target,'mode':mode,'frozen_methods':frozen_methods,'raw_sha256':table_hash(values,names),'chunks':tags}
    return frame_pack(meta,parts)

def decode(frame):
    meta,parts=frame_unpack(frame);n=int(meta['n']);sources=meta['sources'];target=meta['target'];mode=meta['mode'];methods=meta['frozen_methods']
    values={s:dec_column(p,methods[s],n) for s,p in zip(sources,parts[:len(sources)])}
    extra=parts[len(sources):]
    if mode=='raw':
        assert len(extra)==1
        values[target]=dec_column(extra[0],methods[target],n)
    elif mode in ('mask_raw','mask_cents','mask_tri'):
        if len(extra)!=2:raise ValueError('bad masks')
        tri=(mode=='mask_tri');mraw=D.decompress(extra[0]);expected=(n+3)//4 if tri else (n+7)//8
        if len(mraw)!=expected:raise ValueError('bad mask byte count')
        code=unpack_mode(mraw,n,tri)
        if np.any(code>2) or (not tri and np.any(code==1)):raise ValueError('invalid mask code')
        pred_a,pred_b=predictors(values,sources)
        words=fbits(pred_a if mode!='mask_cents' else pred_b)
        if tri:words=np.where(code==1,fbits(pred_b),words)
        exceptions=as_u8(D.decompress(extra[1]))
        if len(exceptions)!=int(np.sum(code==2)):raise ValueError('wrong exception count')
        words[code==2]=exceptions
        values[target]=words.view('<f8').copy()
    elif mode=='xor_raw':
        if len(extra)!=1:raise ValueError('bad xor format')
        words=as_u8(unsplit8(D.decompress(extra[0])))
        if len(words)!=n:raise ValueError('bad xor values')
        a,_=predictors(values,sources)
        values[target]=(words^fbits(a)).view('<f8').copy()
    else:raise ValueError('unrecognized mode')
    if table_hash(values,sources+[target])!=meta['raw_sha256']:raise ValueError('checksum mismatch')
    return values


In [ ]:
# Local algorithmic smoke test only. NEVER call these data "real dataset results".
words=np.array([0x0000000000000000,0x8000000000000000,0x7ff0000000000000,0xfff0000000000000,0x7ff8000000000001,0x7ff80000000000ab,0x0000000000000001,0x7fefffffffffffff],dtype='<u8')
odd=words.view('<f8').copy(); parents={'a':odd,'b':np.array([1.,-2.,3.,4.,5.,6.,7.,8.],dtype='<f8')}
y=odd.copy();values={**parents,'y':y}
for mode in ['raw','mask_raw','mask_cents','mask_tri','xor_raw']:
    methods={k:'zstd' for k in values}
    buf=encode(values,['a','b'],'y',mode,methods)
    restored=decode(buf)
    assert all(fbits(restored[k]).tobytes()==fbits(values[k]).tobytes() for k in values)
    try:decode(buf[:-1]);raise AssertionError('truncation was accepted')
    except (ValueError,zstd.ZstdError,lzma.LZMAError):pass
print('PASS: all five modes, signed zeros, payload-distinct NaNs, infinities, subnormal, truncated frames')


In [ ]:
# Frozen chronological / source-order calibration (first 25%), held-out 75%.
MODES=['raw','mask_raw','mask_cents','mask_tri','xor_raw']
results=[]; raw_modes=[]; negatives=[]; by_dataset=[]

def subset(values,a,b):return {k:v[a:b].copy() for k,v in values.items()}

def joint_frame(values,names,order):
    columns=[as_f8(values[k]) for k in names]
    n=len(columns[0]);mat=np.column_stack(columns)
    blob=mat.tobytes(order='C' if order=='row' else 'F')
    meta={'schema':'SG93-joint-v1','n':n,'names':names,'order':order,'raw_sha256':table_hash(values,names),'chunks':[{'kind':'joint','nbytes':0}]}
    zipped=C.compress(blob);meta['chunks'][0]['nbytes']=len(zipped)
    return frame_pack(meta,[zipped])

def joint_decode(p):
    meta,pieces=frame_unpack(p)
    raw=D.decompress(pieces[0]);n=meta['n'];names=meta['names'];m=len(names)
    if len(raw)!=8*n*m:raise ValueError('joint wrong raw size')
    values=np.frombuffer(raw,dtype='<f8').reshape((n,m),order='C' if meta['order']=='row' else 'F')
    mapp={name:values[:,i].copy() for i,name in enumerate(names)}
    if table_hash(mapp,names)!=meta['raw_sha256']:raise ValueError('joint checksum mismatch')
    return mapp

for data in DATA:
    fam=data['family'];sources=data['sources'];target=data['target'];names=sources+[target]
    m=data['values'];n=len(m[target]);cut=max(50,int(n*.25))
    if n-cut<100:
        ERR.append({'family':fam,'stage':'evaluation','error':'insufficient held-out rows'});continue
    train=subset(m,0,cut);hold=subset(m,cut,n)
    codec_start=time.perf_counter();frozen={}; trials={}
    for key in names:frozen[key],trials[key]=frozen_codec(train[key])
    mode_stats=[]
    for mode in MODES:
        p=encode(train,sources,target,mode,frozen)
        assert all(as_f8(decode(p)[k]).tobytes()==as_f8(train[k]).tobytes() for k in names)
        mode_stats.append({'mode':mode,'cal_bytes':len(p)})
    chosen=min(mode_stats,key=lambda x:x['cal_bytes'])['mode'];cal_seconds=time.perf_counter()-codec_start
    for rec in mode_stats:raw_modes.append({'family':fam,**rec,'chosen':rec['mode']==chosen})
    measured={}
    for mode in MODES:
        t0=time.perf_counter();p=encode(hold,sources,target,mode,frozen);enc_s=time.perf_counter()-t0
        t0=time.perf_counter();decoded=decode(p);dec_s=time.perf_counter()-t0
        bit_ok=all(as_f8(decoded[k]).tobytes()==as_f8(hold[k]).tobytes() for k in names)
        if not bit_ok:raise AssertionError(f'{fam}: {mode}: reconstructed bits differ')
        measured[mode]=len(p)
        states=masks(hold,sources,target,mode) if mode.startswith('mask_') else None
        results.append({'family':fam,'mode':mode,'frozen_selected':mode==chosen,'heldout_bytes':len(p),'bits_verified':bit_ok,'encode_s':enc_s,'decode_s':dec_s,'exact_pred_fraction':float(np.mean(states!=2)) if states is not None else None,'exception_words':int(np.sum(states==2)) if states is not None else None})
    joint={}
    for order in ['row','column']:
        p=joint_frame(hold,names,order)
        assert all(as_f8(joint_decode(p)[k]).tobytes()==as_f8(hold[k]).tobytes() for k in names)
        joint[order]=len(p)
    native_bytes=None
    if pcs is not None:
        try:
            native_frame=encode(hold,sources,target,'raw',{k:'pcodec' for k in names})
            assert all(as_f8(decode(native_frame)[k]).tobytes()==as_f8(hold[k]).tobytes() for k in names)
            native_bytes=len(native_frame)
        except Exception as e:
            ERR.append({'family':fam,'stage':'native_pcodec_baseline','error':repr(e)})
    else:
        ERR.append({'family':fam,'stage':'native_pcodec_baseline','error':'not installed'})
    strong=min(measured['raw'],*joint.values(),*([native_bytes] if native_bytes is not None else []))
    winner=min(measured['raw'],joint['row'],joint['column'])
    size=measured[chosen]
    # Same trained policy: destroy only the tested relation by permuting target values.
    rng=np.random.default_rng(9301);control={k:v.copy() for k,v in hold.items()}
    control[target]=control[target][rng.permutation(len(control[target]))]
    ctrl_p=encode(control,sources,target,chosen,frozen)
    assert all(as_f8(decode(ctrl_p)[k]).tobytes()==as_f8(control[k]).tobytes() for k in names)
    controls={'family':fam,'selected_mode':chosen,'matched_original_bytes':size,'shuffled_target_bytes':len(ctrl_p),'source_order_preserved':True,'negative_control_seed':9301}
    negatives.append(controls)
    by_dataset.append({'family':fam,'n_cal':cut,'n_hold':n-cut,'preselected_mode':chosen,'preselected_bytes':size,'frozen_univariate_bytes':measured['raw'],'zstd_joint_row_bytes':joint['row'],'zstd_joint_column_bytes':joint['column'],'pcodec_native_bytes':native_bytes,'strongest_tested_baseline_oracle_bytes':strong,'saving_pct_vs_strongest':100*(strong-size)/strong,'passes_size_gate':size<=0.95*strong,'calibration_seconds':cal_seconds,'input_sha256':table_hash(hold,names)})
    print(fam,'calibrated',chosen,'held-out',size,'strongest tested comparator (conservative oracle)',strong,'saving%',round(100*(strong-size)/strong,3))


In [ ]:
# Audit conclusion: no novelty is claimed from a positive size result alone.
def writecsv(name,rows):
    fields=list(dict.fromkeys(k for rec in rows for k in rec)) if rows else ['status']
    with (OUT/name).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows if rows else [{'status':'NO_ROWS'}])

for file,rows in [('dataset_provenance.csv',PROVENANCE),('mode_calibration.csv',raw_modes),('heldout_modes.csv',results),('family_summary.csv',by_dataset),('shuffled_controls.csv',negatives),('run_issues.csv',ERR)]:writecsv(file,rows)
sources_complete=len(by_dataset)==2 and not any(r['stage']=='data_loading' for r in ERR)
native_available=pcs is not None and sources_complete and all(x['pcodec_native_bytes'] is not None for x in by_dataset)
complete=sources_complete and native_available
size_pass=complete and all(x['passes_size_gate'] for x in by_dataset)
neg_pass=complete and all(x['selected_mode']!='raw' and x['shuffled_target_bytes']>x['matched_original_bytes'] for x in negatives)
DECISION={'notebook':'NB93','status':'COMPLETE' if complete else 'INCOMPLETE_BASELINE_OR_SOURCE','datasets_evaluated':len(by_dataset),'native_pcodec_available':native_available,'dataset_names':[x['family'] for x in by_dataset],
 'calibration_chosen_before_heldout':True,'bitwise_test_passed':True,'five_percent_size_gate_passed':bool(size_pass),'negative_control_passed':bool(neg_pass),
 'prior_art_NB93_variants_covered_by_existing_dependency_exception_and_XOR_schemes':True,
 'novel_algorithm_established':False,'journal_ready':False,
 'next_action':('REJECT SIZE HYPOTHESIS FOR THESE REAL DATA' if complete and not size_pass else 'SEARCH FOR A DISTINCT THEORETICAL MECHANISM, DO NOT CLAIM THIS IS NOVEL' if size_pass else 'FIX SOURCE OR EVALUATION INCOMPLETENESS'),
 'warnings':['Results are on previously known functional-dependency examples, not independent innovation confirmation.','NYC and bike data are different schemas: no stream-level pooling.','Strongest joint baseline is selected post-test as a deliberately conservative comparator oracle, not a deployable selector.', 'If PCodec is unavailable or fails bitwise validation on any family, the native baseline gate is INCOMPLETE even if all other measurements succeed.','Portable representation reconstructs parsed float64 streams, NOT CSV or Parquet original bytes.','NaN/payload test in self-contained smoke is synthetic correctness evidence only.','Prior-art equivalent baseline such as Corra/Virtual must be directly benchmarked before innovation claims.']}
(OUT/'decision.json').write_text(json.dumps(DECISION,indent=2))
(OUT/'environment.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'zstandard':zstd.__version__,'pcodec_available':pcs is not None,'install':INSTALL},indent=2,default=str))
ARCHIVE=ROOT/'SGX_NB93_results.zip'
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print(json.dumps(DECISION,indent=2));print('Archive:',ARCHIVE)
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except ImportError:
    print('Run on Colab for auto-download; file saved locally.')
